In [6]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


In [9]:
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

In [10]:
sys.path.insert(0, str(PROJECT_ROOT))

In [11]:
notes = pd.read_csv(DATA_DIR / "raw" / "clinical_notes.csv")
print("Original notes:", len(notes))
print(notes.columns.tolist())

Original notes: 1602
['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']


In [12]:
# Remove broken notes
notes_clean = notes[
    notes["clean_note_text"].astype(str).str.strip() != "#NAME?"
].copy()

print("After #NAME? removal:", len(notes_clean))

After #NAME? removal: 1595


In [13]:
# Deduplicate repeated note content per patient
notes_dedup = (
    notes_clean
    .sort_values(["person_id", "creation_timestamp"])
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("After deduplication:", len(notes_dedup))
print("Patients:", notes_dedup["person_id"].nunique())

After deduplication: 1103
Patients: 50


In [14]:
print(notes_dedup.columns.tolist())

['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']


In [20]:
# ============================================================
# SECTION-AWARE CHUNKING
#
# Split each cleaned clinical note into its existing sections.
# Each detected section becomes an independent retrieval chunk
# while preserving patient and chronological metadata.
# ============================================================

import re

import pandas as pd


def split_note_into_sections(note_text):
    """
    Split a clinical note using explicit section headings already
    present in the note.

    Returns:
        list of dictionaries:
        [
            {
                "section_name": "...",
                "chunk_text": "..."
            },
            ...
        ]
    """

    text = str(note_text).strip()

    if not text:
        return []

    # Common clinical section headings.
    section_names = [
        "Chief Complaint",
        "Presenting Complaint",
        "History of Present Illness",
        "HPI",
        "Past Medical History",
        "PMH",
        "Past Surgical History",
        "PSH",
        "Medications",
        "Current Medications",
        "Allergies",
        "Family History",
        "Social History",
        "Review of Systems",
        "ROS",
        "Physical Examination",
        "Physical Exam",
        "Examination",
        "Vital Signs",
        "Vitals",
        "Investigations",
        "Laboratory Results",
        "Labs",
        "Imaging",
        "Assessment",
        "Impression",
        "Diagnosis",
        "Diagnoses",
        "Plan",
        "Assessment and Plan",
        "Treatment",
        "Hospital Course",
        "Clinical Course",
        "Discharge Plan",
        "Follow Up",
        "Follow-Up",
    ]

    heading_pattern = "|".join(
        re.escape(name)
        for name in sorted(
            section_names,
            key=len,
            reverse=True
        )
    )

    # Recognize headings such as:
    # Assessment:
    # ASSESSMENT
    # Plan -
    pattern = re.compile(
        rf"(?im)^[ \t]*(?P<heading>{heading_pattern})"
        rf"[ \t]*(?::|-)?[ \t]*$"
    )

    matches = list(pattern.finditer(text))

    # If the note contains no recognizable section headings,
    # preserve the complete note as one chunk rather than
    # discarding information.
    if not matches:
        return [{
            "section_name": "Unsectioned",
            "chunk_text": text
        }]

    sections = []

    # Preserve text appearing before the first recognized heading.
    prefix = text[:matches[0].start()].strip()

    if prefix:
        sections.append({
            "section_name": "Preamble",
            "chunk_text": prefix
        })

    # Extract each section.
    for i, match in enumerate(matches):

        section_name = match.group(
            "heading"
        ).strip()

        content_start = match.end()

        if i + 1 < len(matches):
            content_end = matches[i + 1].start()
        else:
            content_end = len(text)

        content = text[
            content_start:content_end
        ].strip()

        if content:
            sections.append({
                "section_name": section_name,
                "chunk_text": (
                    f"{section_name}\n{content}"
                )
            })

    return sections

In [21]:
# ------------------------------------------------------------
# Create one retrieval row per clinical section.
# ------------------------------------------------------------

section_records = []

for _, row in notes_dedup.iterrows():
    sections = split_note_into_sections(
        row["clean_note_text"]
    )

    for section in sections:

        section_records.append({
            "person_id": row["person_id"],
            "creation_timestamp": row[
                "creation_timestamp"
            ],
            "section_name": section[
                "section_name"
            ],
            "chunk_text": section[
                "chunk_text"
            ],
        })


section_chunks = pd.DataFrame(
    section_records
)

# Preserve chronological ordering.
section_chunks = (
    section_chunks
    .sort_values(
        [
            "person_id",
            "creation_timestamp",
        ]
    )
    .reset_index(drop=True)
)

# Unique chunk identifier.
section_chunks["chunk_id"] = range(
    len(section_chunks)
)

print(
    "Total section chunks:",
    len(section_chunks)
)

print(
    "Patients:",
    section_chunks["person_id"].nunique()
)

print("\nSection counts:")
print(
    section_chunks[
        "section_name"
    ].value_counts().head(20)
)

display(section_chunks.head(10))

Total section chunks: 2771
Patients: 50

Section counts:
section_name
Unsectioned             684
Preamble                419
Plan                    405
Investigations          369
Presenting Complaint    364
Medications              91
Past Medical History     80
Allergies                80
Family History           79
Social History           78
Impression               58
Review of Systems        29
Diagnosis                22
Physical Examination      7
Physical examination      2
Assessment                2
Examination               1
Current Medications       1
Name: count, dtype: int64


,person_id,creation_timestamp,section_name,chunk_text,chunk_id
0,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 08:45,Unsectioned,"- Patient: Tomos Ellis, 15-year-old male, pres...",0
1,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:10,Unsectioned,"Patient: Tomos Ellis, 15-year-old male, presen...",1
2,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:25,Unsectioned,"Patient name: Tomos Ellis, 15-year-old male. N...",2
3,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:00,Unsectioned,"Reviewed abdominal X-ray on 2026-01-07, which ...",3
4,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Preamble,Patient\nTomos Ellis\n\nAge\n15\n\nSex\nMale\n...,4
5,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Presenting Complaint,Presenting Complaint\nAbdominal pain\n\nHistor...,5
6,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Past Medical History,Past Medical History\nNil,6
7,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Medications,Medications\nNil,7
8,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Allergies,Allergies\nNKDA,8
9,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Social History,Social History\nLives at home with parents. At...,9


In [16]:
# Load the BGE model for embeddings

bge_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

print("BGE model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE model loaded.


In [31]:
# ------------------------------------------------------------
# Remove exact duplicate section content within each patient.
# Keep the earliest occurrence so chronology is preserved.
# ------------------------------------------------------------

section_chunks["normalized_chunk_text"] = (
    section_chunks["chunk_text"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

section_chunks = (
    section_chunks
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "normalized_chunk_text"
        ],
        keep="first"
    )
    .drop(
        columns=["normalized_chunk_text"]
    )
    .reset_index(drop=True)
)

section_chunks["chunk_id"] = range(
    len(section_chunks)
)

print(
    "Section chunks after exact deduplication:",
    len(section_chunks)
)

Section chunks after exact deduplication: 2596


In [32]:
# ============================================================
# Generate BGE embeddings for all section-aware chunks.
#
# These embeddings form the retrieval index for the final
# Section + BGE RAG workflow.
# ============================================================

section_texts = (
    section_chunks["chunk_text"]
    .astype(str)
    .tolist()
)

section_embeddings = bge_model.encode(
    section_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(
    "Section embeddings shape:",
    section_embeddings.shape
)

Batches:   0%|          | 0/82 [00:00<?, ?it/s]

Section embeddings shape: (2596, 768)


In [33]:
# ------------------------------------------------------------
# Save the reusable Section + BGE embedding index.
# ------------------------------------------------------------

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "rag"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(
    str(OUTPUT_DIR / "section_bge_embeddings.npy"),
    section_embeddings
)

section_chunks.to_csv(
    str(OUTPUT_DIR / "section_chunks.csv"),
    index=False
)

print("Section chunks and BGE embeddings saved.")

Section chunks and BGE embeddings saved.


In [34]:
# ============================================================
# SECTION + BGE RETRIEVAL
#
# 1. Restrict retrieval to one patient
# 2. Embed the query with BGE
# 3. Calculate cosine similarity
# 4. Select Top-20 section chunks
# 5. Reorder retrieved chunks chronologically
# ============================================================

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def retrieve_section_bge(
    patient_id,
    query,
    top_k=20
):

    # --------------------------------------------------------
    # 1. Find chunks belonging only to this patient
    # --------------------------------------------------------

    patient_mask = (
        section_chunks["person_id"] == patient_id
    )

    patient_chunks = (
        section_chunks.loc[patient_mask]
        .copy()
    )

    if patient_chunks.empty:
        raise ValueError(
            f"No section chunks found for patient: {patient_id}"
        )

    # Use the same rows from the embedding matrix.
    patient_embeddings = section_embeddings[
        patient_mask.to_numpy()
    ]


    # --------------------------------------------------------
    # 2. Embed retrieval query
    # --------------------------------------------------------

    query_embedding = bge_model.encode(
        [query],
        normalize_embeddings=True
    )


    # --------------------------------------------------------
    # 3. Cosine similarity
    # --------------------------------------------------------

    similarities = cosine_similarity(
        query_embedding,
        patient_embeddings
    )[0]


    patient_chunks["similarity"] = similarities


    # --------------------------------------------------------
    # 4. Select Top-K most relevant sections
    # --------------------------------------------------------

    retrieved = (
        patient_chunks
        .sort_values(
            "similarity",
            ascending=False
        )
        .head(top_k)
        .copy()
    )

    # Keep retrieval rank before chronological reordering.
    retrieved["retrieval_rank"] = range(
        1,
        len(retrieved) + 1
    )


    # --------------------------------------------------------
    # 5. Reorder retrieved evidence chronologically
    #    before sending it to the summarization model.
    # --------------------------------------------------------

    retrieved = (
        retrieved
        .sort_values(
            [
                "creation_timestamp",
                "chunk_id"
            ]
        )
        .reset_index(drop=True)
    )

    return retrieved

In [35]:
PATIENT_ID = "04df53ea-55c1-48d9-84a1-1f15c133b29b"

RAG_QUERY = """
Retrieve the clinically important information needed to create a
longitudinal patient summary, including presenting problems, diagnoses,
symptoms, investigations, treatments, medications, clinical progression,
outcomes, and follow-up plans.
"""

retrieved_sections = retrieve_section_bge(
    patient_id=PATIENT_ID,
    query=RAG_QUERY,
    top_k=20
)

print("Retrieved sections:", len(retrieved_sections))

display(
    retrieved_sections[
        [
            "creation_timestamp",
            "section_name",
            "retrieval_rank",
            "similarity",
            "chunk_text"
        ]
    ]
)

Retrieved sections: 20


,creation_timestamp,section_name,retrieval_rank,similarity,chunk_text
0,01/01/2026 07:30,Unsectioned,11,0.521042,"67M, Allan Victor Robinson (DOB: 1956-03-15, N..."
1,01/01/2026 08:00,Unsectioned,9,0.522843,67M. GCS 12/15 - disoriented to time/place. BP...
2,01/01/2026 08:30,Unsectioned,2,0.552219,"- Patient: Allan Victor R obinson, 67-year-old..."
3,01/01/2026 10:45,Presenting Complaint,14,0.519568,Presenting Complaint\nAcute confusion followin...
4,01/01/2026 10:45,Past Medical History,4,0.547768,Past Medical History\n- HTN\n- OA
5,01/01/2026 11:30,Plan,12,0.520066,Plan\n- Start paracetamol 1g QDS PO for headac...
6,01/01/2026 15:30,Unsectioned,19,0.514177,Physiotherapy assessment conducted on 01/01/26...
7,01/01/2026 16:00,Unsectioned,13,0.519738,NOK Maargaret Robinson phoned at 16:00. Update...
8,01/01/2026 17:00,Unsectioned,16,0.517933,Occupational therapy assessment conducted on 0...
9,02/01/2026 10:30,Unsectioned,18,0.514246,Physiotherapy session conducted on 2026-01-02 ...


In [36]:
# ============================================================
# BUILD CHRONOLOGICAL RAG CONTEXT
#
# Convert the Top-20 retrieved Section + BGE chunks into
# chronological evidence for longitudinal summarization.
# ============================================================

rag_context_parts = []

for _, row in retrieved_sections.iterrows():

    rag_context_parts.append(
        f"[{row['creation_timestamp']}] "
        f"[{row['section_name']}]\n"
        f"{row['chunk_text']}"
    )

rag_context = "\n\n".join(
    rag_context_parts
)

print("Retrieved chunks:", len(retrieved_sections))
print("Context characters:", len(rag_context))
print("Context words:", len(rag_context.split()))

print("\n--- CONTEXT PREVIEW ---\n")
print(rag_context[:3000])

Retrieved chunks: 20
Context characters: 10815
Context words: 1549

--- CONTEXT PREVIEW ---

[01/01/2026 07:30] [Unsectioned]
67M, Allan Victor Robinson (DOB: 1956-03-15, NHS: 680034032), arrived A&E 07:30 w/ acute confusion post minor fall. Triage by Nurse Shaun Patrick Hopkins. Vitals: BP 170/95, HR 88. No allergies. PMHx: HTN, OA. No meds. Assigned Cat 2 due to confusion & recent trauma. ED dx: Subdural hygroma. Preped for further assess & invx. Awaiting revi ew.
Nurse Shaun Patrick Hopkins 
NMC number: 05J2149V

[01/01/2026 08:00] [Unsectioned]
67M. GCS 12/15 - disoriented to time/place. BP 170/95, pulse 88 bpm. Triaged by Nurse Shaun Patrick Hopkins. Chief complaint: acute confusion after minor fall. Triage categody: 2. PMH: HTN, Osteoarthritis. No known drug allergies. Referred for urgent bloods  + CT head to r/o ICI.
Nurse Shaun Patrick Hopkins 
NMC number: 05J2149V

[01/01/2026 08:30] [Unsectioned]
- Patient: Allan Victor R obinson, 67-year-old male, NHS number 680034032. 
 - D

In [ ]:

RAG_QUERY = """
Retrieve the clinically relevant information needed to produce a
comprehensive longitudinal summary of this patient's clinical history,
including major diagnoses, treatments, investigations, clinical
progression, and outcomes.
""".strip()

In [ ]:
# ============================================================
# GENERATE FINAL SECTION + BGE RAG SUMMARY
# ============================================================

from config.prompts import SUMMARY_PROMPT
from src.llm.llm import generate_rag_summary

section_bge_summary = generate_rag_summary(
    context=rag_context,
    prompt=SUMMARY_PROMPT
)

print(section_bge_summary)